In [16]:
# Etape 1 - explorer sans pandas
import json

with open("../data/villes.json", encoding="utf-8") as f:
    data = json.load(f)

print(f"nombre de villes : {len(data["villes"])}")
print(f"nom de la deuxième ville : {data["villes"][1]["nom"]}")
print(f"nombre de restaurants à Syracuse : {len(data["villes"][2]["restaurants"])}")
print(f"nom du premier restaurant de Palerme : {data["villes"][1]["restaurants"][0]["nom"]}")

nombre de villes : 3
nom de la deuxième ville : Palerme
nombre de restaurants à Syracuse : 2
nom du premier restaurant de Palerme : Antica Focacceria


In [18]:
# Etape 2 - compter à la main
for v in data["villes"]:
    nb_resto = len(v["restaurants"])
    print(f"A {v["nom"]}, il y a {nb_resto} restaurants")

A Catane, il y a 3 restaurants
A Palerme, il y a 2 restaurants
A Syracuse, il y a 2 restaurants


In [97]:
# Etape 3 - tout mettre à plat
import pandas as pd

df = pd.json_normalize(data,
    record_path=["villes", "restaurants"],
    meta=[["villes", "nom"], ["villes", "region"], ["villes", "habitants"]])

df = df.rename(columns={"villes.nom": "ville", "villes.region": "region", "villes.habitants": "habitants"})
print(df)

                             nom       type                 notes     ville  \
0  Gelateria Zio Pietro dal 1964    glacier       [4.6, 4.8, 4.7]    Catane   
1                 A Casa d'Amici  trattoria            [4.4, 4.5]    Catane   
2                       La Bitta    poisson  [4.2, 4.0, 4.3, 4.1]    Catane   
3              Antica Focacceria  trattoria       [4.3, 4.1, 4.4]   Palerme   
4               Ferro di Cavallo  trattoria            [4.5, 4.6]   Palerme   
5                    Don Camillo    poisson  [4.7, 4.8, 4.6, 4.9]  Syracuse   
6               Gelateria Bianca    glacier                 [4.2]  Syracuse   

   region habitants  
0  Sicile    298762  
1  Sicile    298762  
2  Sicile    298762  
3  Sicile    630828  
4  Sicile    630828  
5  Sicile    121605  
6  Sicile    121605  


In [101]:
# Etape 4 - exploiter
df["note_moyenne"] = df["notes"].apply(lambda x: round(sum(x) / len(x), 2) if isinstance(x, list) and len(x) > 0 else None)

df_final = df.drop(columns="notes")

print(df_final)
print()

note_max = df_final["note_moyenne"].max()
print(f"Question 1 : {df_final["nom"][df_final["note_moyenne"] == note_max].to_string(index=False)} a la meilleure note moyenne")
print()

print(f"Question 2 : Nombre de types de restaurants :")
print(f"{df_final["type"].value_counts().to_string(header=False)}")


                             nom       type     ville  region habitants  \
0  Gelateria Zio Pietro dal 1964    glacier    Catane  Sicile    298762   
1                 A Casa d'Amici  trattoria    Catane  Sicile    298762   
2                       La Bitta    poisson    Catane  Sicile    298762   
3              Antica Focacceria  trattoria   Palerme  Sicile    630828   
4               Ferro di Cavallo  trattoria   Palerme  Sicile    630828   
5                    Don Camillo    poisson  Syracuse  Sicile    121605   
6               Gelateria Bianca    glacier  Syracuse  Sicile    121605   

   note_moyenne  
0          4.70  
1          4.45  
2          4.15  
3          4.27  
4          4.55  
5          4.75  
6          4.20  

Question 1 : Don Camillo a la meilleure note moyenne

Question 2 : Nombre de types de restaurants :
trattoria    3
glacier      2
poisson      2
